<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-header">

## Tutorial \#3: Neurons

</div>

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Neurons are the core of every neural network and who are we to change that. In <b><i>Spark</i></b> neurons are defined through the class <b>spark.nn.Neuron</b>, this module has the sole purpose of gather and organize the computation of the smaller and modular components into a cohesive unit.

In order to build a successful neuron, it may be necessary to shift the mindset from bricklayer to architect. Unlike other frameworks, in order to fully embrace the modular design mindset it is necessary to define neurons as blueprints, rather than fixed objects. This is achieved by filling a series of templates, the <b>spark.ModuleSpecs</b>. This object expects a few inputs: a name for the module, the module we want to create, its associated configuration class and an its relationships with other modules.

Let's start by constructing the most simple neuron possible: a Leaky-and-Integrate neuron.

⚠️ WARNING: The following may look scary at first, but it becomes incredibly intuitive later on, not to mention useful. 

</div>

In [ ]:
# Imports
import spark
import jax
import jax.numpy as jnp

# Linear synapses
synapses_spec = spark.ModuleSpecs(
	name ='synapses', 															# <--- Defining a module named 'synapses'
	module_cls = spark.nn.synapses.LinearSynapses, 								# <--- The module is of type LinearSynapses
	inputs = {																	# <--- We need to indicate how all inputs to the module are served
		'spikes': [																# <--- This module has only one expected input: "spikes"
            spark.PortMap(origin='__call__', port='incoming_spikes')			# <--- Mappings are defined through PortMap's, which are simply a pair
		]																		#      of labels, origin or name of the module producing the input and 
	},																			#      port which is the name of the output variable produced by the input
)																				#      module

# Leaky soma
soma_spec = spark.ModuleSpecs(
	name ='soma', 																# <--- Defining a module named 'soma'
	module_cls = spark.nn.somas.LeakySoma, 										# <--- The module is of type LeakySoma
	inputs = {																	# <--- We need to indicate how all inputs to the module are served
		'current': [															# <--- This module has one expected input: "current"
            spark.PortMap(origin='synapses', port='currents')					# <--- The module defined above has one expected output: "currents"
		],
		'inhibition_mask': [													# <--- This module has one optional input: "inhibition_mask"
            spark.PortMap(
                origin='__self__', 												# <--- Neurons by default define a property called: "inhibition_mask"
                port='inhibition_mask', 
                is_property=True
			)
		],
	},
	outputs = {																	# <--- "outputs" is used to define an output for the entire model.
		'my_awesome_spikes': 'spikes', 											#      in this case, we define a variable "my_awesome_spikes" from the expected
	},																			#      output of the model "spikes"
)

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Finally, we just need to wrap everything into the usual Config / Object pattern.
That's it, a new neuron model out from a simple blueprint. 

❗ Spoiler: We will be able to edit this model later on with more powerful tools!.

</div>

In [2]:
@spark.register_config								# <--- Register the configuration object
class MyAwesomeLIFNeuronConfig(spark.nn.NeuronConfig):
	
	modules_specs: list[spark.ModuleSpecs]  = [		# <--- Now, we simply add every spec in a list under the config name "modules_specs"
			synapses_spec,
			soma_spec
		]

@spark.register_neuron								# <--- Register the neuron object
class MyAwesomeLIFNeuron(spark.nn.Neuron):
	config: MyAwesomeLIFNeuronConfig

<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

Now, let's try it!

</div>

In [3]:
# Model initialization
units = (8,)
input_shape = (16,)
alif_neurons = MyAwesomeLIFNeuron(
    units=units,
	inhibitory_rate=0.2,
)

# Test the model.
rng = jax.random.key(42)
rng, key = jax.random.split(rng, 2)
in_spikes = spark.SpikeArray( 
    jax.random.uniform(key, shape=input_shape, dtype=jnp.float16) < 0.5 
)
out_spikes = alif_neurons(incoming_spikes=in_spikes)		# <--- Remember, we told the model to expect a variable "incoming_spikes"
print(out_spikes, '\n')										#      and to output "my_awesome_spikes"


# Let's test this puppy
@jax.jit
def run_model_split(graph, state, **inputs):
	model = spark.merge(graph, state)
	outputs = model(**inputs)
	_, state = spark.split((model))
	return outputs, state

graph, state = spark.split((alif_neurons))
outputs, state = run_model_split(graph, state, incoming_spikes=in_spikes)

%timeit run_model_split(graph, state, incoming_spikes=in_spikes)

{'my_awesome_spikes': SpikeArray(_encoding=Array([0, 2, 0, 0, 0, 0, 0, 0], dtype=uint8), async_spikes=False)} 

38.9 μs ± 1.16 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

However, if this approach does not serve your needs and perhaps you really really REALLY want to implement something that is quite hard to decompose on simple modular operations you still can implement a neuron in a more classical way by means of <b>spark.nn.Module</b>. However, doing so prevents the model from being fully reusable across the rest of the framework.

Implementing a neuron this way is quite straight forward, and very similar to the first tutorial. The only difference is that you will need to manually define how the computation is forwarded between components. To illustrate this, let's recreate the previous neuron using modules. 

As usual we start with a blueprint of what we need and what we are going to get.

</div>

In [4]:
import sys
sys.path.insert(1, './..')


# Imports
import spark
import jax
import jax.numpy as jnp


import typing as tp

class NeuronOutput(tp.TypedDict):
    out_spikes: spark.SpikeArray
    
@spark.register_config
class MyCustomLIFNeuronConfig(spark.nn.Config):

	units: tuple[int, ...]
	inhibitory_rate: float = 0.2
	soma: spark.nn.somas.LeakySomaConfig 					# <--- Using components inside another Module requires 
	synapses: spark.nn.synapses.LinearSynapsesConfig 		#      you to also inform the configuration object

Now, this is where things get a little bit rough. Usually <b><i>Spark</i></b> manages several aspect of the initialization automatically; standard specifications allow us to greatly reduce what needs to be said. However, when creating a custom module and, in particular, a custom neuron, a lot of things that we would like to hear need to be said, <b>explicitly</b>. It is impossible for <b><i>Spark</i></b> to predict user intention!.

In [ ]:
from math import prod
from spark.core.utils import validate_shape

@spark.register_module
class MyCustomLIFNeuron(spark.nn.Module):
	config: MyCustomLIFNeuronConfig

	def __init__(self, config: MyCustomLIFNeuronConfig | None = None, **kwargs):
		super().__init__(config=config, **kwargs)
		# Extract the units from the configuration
		self.units = validate_shape(self.config.units)		# <--- NOTE: Always use the self.config never the config!
		self._units = prod(self.units)
		# Initialize inhibitory mask.
		inhibitory_units = int(self._units * self.config.inhibitory_rate)
		indices = jax.random.permutation(self.get_rng_keys(1), jnp.arange(self._units), independent=True)[:inhibitory_units]
		inhibition_mask = jnp.zeros((self._units,), dtype=jnp.bool)
		inhibition_mask = inhibition_mask.at[indices].set(True).reshape(self.units)
		self._inhibition_mask = spark.Constant(inhibition_mask, dtype=jnp.bool)
		

	# NOTE: The following is not necessary, but by design spark.nn.Neuron expose the inhibitory_mask 
	# used to split  the neuron pool into excitatory and inhibitory units.
	@spark.property
	def inhibition_mask(self,) -> spark.BooleanMask:
		return spark.BooleanMask(self._inhibition_mask.value)

	# NOTE: Similarly, this is not necessary unless you want to allow self connections inside this pool.
	# In general, any module that may connect to itself needs to indicate what it outputs in greater detail.
	def recurrent_contract(
			self, 
		) -> tuple[dict, dict]:
		# Output specs
		output_contract_specs = {k:v['spec'] for k,v in self._get_output_specs().items()}
		output_contract_specs['out_spikes'].shape = self.units
		output_contract_specs['out_spikes'].inhibition_mask = self._inhibition_mask
		# Property specs. Properties should be defined inside __init__, so it is safe to inspect them.
		property_contract_specs = self._get_property_specs()
		property_contract_specs['inhibition_mask'] = self.inhibition_mask
		return output_contract_specs, property_contract_specs

	# NOTE: You may use "abc_args" to perform a more complex initialization, in this case it is not really necessary.
	def build(self, **abc_args: spark.SparkPayload):
		# Soma model.
		self.soma = self.config.soma.class_ref(config=self.config.soma)
		# Synaptic model.
		self.synapses = self.config.synapses.class_ref(config=self.config.synapses)


	# The __call__ method is the soul of every module. As we saw in the first tutorial, it has three important elements.
	# 1) What we expect to get at the begining, which need to be fully anotated, (e.g. in_spikes: spark.SpikeArray)
	# 2) What we expect to get at the end (e.g. NeuronOutput)
	# 3) The actual computation
	def __call__(self, incoming_spikes: spark.SpikeArray) -> NeuronOutput:
		"""
			Update neuron's states and compute spikes.
		"""
		# Pass in_spikes to the synapses
		synapses_output = self.synapses(spikes=in_spikes)

		# Synapses returns a dictionary with all its outputs, which in this case its only "currents".
		synaptic_currents = synapses_output['currents']

		# Soma components accept an optional parameter, "inhibition_mask" that is used to produce a signed spikes. 
		# If this parameter is passed or manually computer later on, Spark will see all these neurons as excitatory neurons.
		soma_output = self.soma(current=synaptic_currents, inhibition_mask=self.inhibition_mask)

		# Construct output. Note that the output of __call__ must always match with the definition of NeuronOutput.
		# This information is used to verify the model at several stages.
		output_spikes = soma_output['spikes']
		return {
			'out_spikes': output_spikes
		}


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

This would be enough to create a copy of the first model we created. Without further ado, let's try all this hard work!

</div>

In [6]:
# Model initialization
units = (8,)
input_shape = (16,)
alif_neurons = MyCustomLIFNeuron(
	_s_units=units,
	inhibitory_rate=0.2,
)

# Test the model.
rng = jax.random.key(42)
rng, key = jax.random.split(rng, 2)
in_spikes = spark.SpikeArray( 
    jax.random.uniform(key, shape=input_shape, dtype=jnp.float16) < 0.5 
)
out_spikes = alif_neurons(incoming_spikes=in_spikes)		# <--- Remember, we told the model to expect a variable "incoming_spikes"
print(out_spikes, '\n')										#      and to output "my_awesome_spikes"


# Let's test this puppy
@jax.jit
def run_model_split(graph, state, **inputs):
	model = spark.merge(graph, state)
	outputs = model(**inputs)
	_, state = spark.split((model))
	return outputs, state

graph, state = spark.split((alif_neurons))
outputs, state = run_model_split(graph, state, incoming_spikes=in_spikes)

%timeit run_model_split(graph, state, incoming_spikes=in_spikes)

{'out_spikes': SpikeArray(_encoding=Array([0, 0, 0, 0, 0, 0, 2, 0], dtype=uint8), async_spikes=False)} 

45.7 μs ± 2.08 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


<link rel="stylesheet" type="text/css" href="./style.css">

<div class="tutorial-text">

As we can see, both models ended up with an almost identical behavior (up to the randomness of the inhibition_mask), and more importantly, with a very similar performance. In our particular case, it turns out that using the controller approach (<b>spark.nn.Neuron</b>), almost always produces a slightly better performing model; the controller approach tends to expose the computation more nicely to the JIT compiler, but that does not mean that it is always going to perform better. Remember: the controller is just a heuristic and is not always the best one!.

</div>